In [3]:
# # edu_qgc_synth_node.py
# # -*- coding: utf-8 -*-
# """
# Node classification experiment on synthetic graph dataset using EDU-QGC.
# - Loads synthetic dataset from CSVs (nodes.csv, features.csv, edges.csv, metadata.json)
# - Runs node classification with train/val/test splits from dataset
# - Keeps EDUQGCNodeClassifier intact
# """

# import os, json, random
# import numpy as np
# import pandas as pd
# import torch
# import torch.nn.functional as F
# from torch import nn
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# from sklearn.manifold import TSNE

# import pennylane as qml
# from torch_geometric.data import Data

# # preprocessing
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

# # =====================================================
# # Reproducibility
# # =====================================================
# def set_seed(seed=42):
#     random.seed(seed); np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed_all(seed)

# # =====================================================
# # Model (unchanged)
# # =====================================================
# from torch.nn import Dropout   # only add Dropout wrapper externally if needed

# class EDUQGCNodeClassifier(nn.Module):
#     def __init__(self, n_nodes, in_feats, T=2, seed=0, use_gpu_qnode=True, use_feat_skip=True, num_classes=7):
#         super().__init__()
#         self.n_nodes = n_nodes
#         self.T = T
#         self.use_feat_skip = use_feat_skip

#         self.enc_W = nn.Parameter(torch.randn(T, 2, in_feats) * 0.08)
#         self.enc_b = nn.Parameter(torch.randn(T, 2) * 0.02)

#         self.edge_phase  = nn.Parameter(torch.randn(T) * 0.08)
#         self.pre_theta   = nn.Parameter(torch.randn(T) * 0.08)
#         self.pre_psi     = nn.Parameter(torch.randn(T) * 0.08)
#         self.post_theta  = nn.Parameter(torch.randn(T) * 0.08)
#         self.post_psi    = nn.Parameter(torch.randn(T) * 0.08)

#         readin_dim = 1 + in_feats if use_feat_skip else 1
#         self.readout = nn.Sequential(
#             nn.Linear(readin_dim, num_classes),
#             Dropout(p=0.3)  # 🔹 dropout for regularization
#         )

#         use_cuda = torch.cuda.is_available()
#         qdev_name = "lightning.gpu" if (use_gpu_qnode and use_cuda) else "default.qubit"
#         self.dev = qml.device(qdev_name, wires=n_nodes, shots=None)

#         @qml.qnode(self.dev, interface="torch", diff_method="best")
#         def circuit(edge_index, X, enc_W, enc_b,
#                     edge_phase, pre_theta, pre_psi, post_theta, post_psi):
#             for t in range(self.T):
#                 enc_out = X @ enc_W[t].T + enc_b[t]
#                 alphas = enc_out[:, 0]; betas = enc_out[:, 1]
#                 for i in range(self.n_nodes):
#                     qml.RX(alphas[i], wires=i)
#                     qml.RY(betas[i], wires=i)

#                 for i in range(self.n_nodes):
#                     qml.RZ(pre_psi[t], wires=i)
#                     qml.RX(pre_theta[t], wires=i)

#                 E = edge_index.shape[1]
#                 for e in range(E):
#                     u = int(edge_index[0, e].item()); v = int(edge_index[1, e].item())
#                     if u != v:
#                         qml.ControlledPhaseShift(edge_phase[t], wires=[u, v])

#                 for i in range(self.n_nodes):
#                     qml.RZ(post_psi[t], wires=i)
#                     qml.RX(post_theta[t], wires=i)

#             return [qml.expval(qml.Z(i)) for i in range(self.n_nodes)]

#         self._circuit = circuit

#     def forward(self, edge_index_torch, x_torch):
#         device = next(self.parameters()).device
#         edge_index = edge_index_torch.to(device)
#         X = x_torch.to(device).float()

#         out = self._circuit(edge_index, X,
#                             self.enc_W, self.enc_b,
#                             self.edge_phase,
#                             self.pre_theta, self.pre_psi,
#                             self.post_theta, self.post_psi)

#         expvals = torch.stack(out, dim=0).float().to(device)
#         if expvals.dim() == 1:
#             expvals = expvals.unsqueeze(1)
#         elif expvals.dim() == 2 and expvals.shape[1] == 1:
#             pass
#         else:
#             expvals = expvals.squeeze(-1).unsqueeze(1)

#         readin = torch.cat([expvals, X], dim=1) if self.use_feat_skip else expvals
#         logits = self.readout(readin)
#         return logits

# # =====================================================
# # Training / Evaluation
# # =====================================================

# def train_nodeclf(model, data, opt):
#     model.train()
#     logits = model(data.edge_index, data.x)
#     loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
#     opt.zero_grad(); loss.backward(); opt.step()
#     return loss.item()

# @torch.no_grad()
# def evaluate_nodeclf(model, data, mask):
#     model.eval()
#     logits = model(data.edge_index, data.x)
#     pred = logits.argmax(1)
#     loss = F.cross_entropy(logits[mask], data.y[mask]).item()
#     acc = accuracy_score(data.y[mask].cpu(), pred[mask].cpu())
#     return loss, acc, pred.cpu().numpy(), logits.detach().cpu().numpy()

# # =====================================================
# # Load synthetic dataset
# # =====================================================

# # --- inside load_synthetic_graph() ---
# from sklearn.model_selection import train_test_split

# def load_synthetic_graph(dataset_dir, train_ratio=0.6, val_ratio=0.2, test_ratio=0.2, seed=42):
#     nodes = pd.read_csv(os.path.join(dataset_dir, "nodes.csv"))
#     edges = pd.read_csv(os.path.join(dataset_dir, "edges.csv"))

#     # Features
#     feature_cols = [c for c in nodes.columns if c not in ["id", "label"]]
#     X = nodes[feature_cols]

#     # Encode categorical
#     cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
#     if cat_cols:
#         for col in cat_cols:
#             le = LabelEncoder()
#             X.loc[:, col] = le.fit_transform(X[col].astype(str))

#     # Normalize numeric
#     num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()
#     if num_cols:
#         scaler = StandardScaler()
#         X.loc[:, num_cols] = scaler.fit_transform(X[num_cols])

#     feats = X.values.astype(np.float32)

#     # Encode labels
#     labels = LabelEncoder().fit_transform(nodes["label"])
#     labels = torch.tensor(labels, dtype=torch.long)

#     edge_index = torch.tensor(edges.values.T, dtype=torch.long)

#     data = Data(x=torch.tensor(feats, dtype=torch.float),
#                 edge_index=edge_index,
#                 y=labels)

#     # --- ✅ stratified train/val/test split ---
#     idx = np.arange(len(labels))
#     y_np = labels.numpy()

#     train_idx, test_idx = train_test_split(idx, stratify=y_np,
#                                            test_size=test_ratio,
#                                            random_state=seed)
#     train_idx, val_idx = train_test_split(train_idx, stratify=y_np[train_idx],
#                                           test_size=val_ratio/(train_ratio+val_ratio),
#                                           random_state=seed)

#     # Masks
#     data.train_mask = torch.zeros(len(labels), dtype=torch.bool); data.train_mask[train_idx] = True
#     data.val_mask   = torch.zeros(len(labels), dtype=torch.bool); data.val_mask[val_idx] = True
#     data.test_mask  = torch.zeros(len(labels), dtype=torch.bool); data.test_mask[test_idx] = True

#     with open(os.path.join(dataset_dir, "metadata.json")) as f:
#         meta = json.load(f)

#     return data, meta




# # =====================================================
# # Run experiment
# # =====================================================
# # --- in run_node_classification() ---
# from sklearn.utils.class_weight import compute_class_weight

# def run_node_classification(dataset_dir, epochs=30, lr=0.03, seed=42):
#     set_seed(seed)
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#     data, meta = load_synthetic_graph(dataset_dir)
#     data = data.to(device)

#     N, F, C = data.num_nodes, data.num_features, len(np.unique(data.y.cpu().numpy()))

#     model = EDUQGCNodeClassifier(n_nodes=N, in_feats=F, num_classes=C).to(device)
#     opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

#     # ✅ compute class weights
#     y_train = data.y[data.train_mask].cpu().numpy()
#     class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
#     class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

#     # replace default loss
#     loss_fn = nn.CrossEntropyLoss(weight=class_weights)

#     tr_losses, val_accs = [], []
#     for ep in range(1, epochs+1):
#         model.train()
#         logits = model(data.edge_index, data.x)
#         loss = loss_fn(logits[data.train_mask], data.y[data.train_mask])
#         opt.zero_grad(); loss.backward(); opt.step()

#         val_loss, val_acc, _, _ = evaluate_nodeclf(model, data, data.val_mask)
#         tr_losses.append(loss.item()); val_accs.append(val_acc)
#         print(f"Epoch {ep:03d} | tr_loss={loss.item():.3f} | val_loss={val_loss:.3f} | val_acc={val_acc:.3f}")

#     # --- test ---
#     te_loss, te_acc, pred, logits = evaluate_nodeclf(model, data, data.test_mask)
#     print("\n[Test] loss={:.3f} acc={:.3f}".format(te_loss, te_acc))

#     # Metrics
#     y_true = data.y[data.test_mask].cpu().numpy()
#     print(classification_report(y_true, pred[data.test_mask.cpu().numpy()]))

#     # --------- Visualizations ---------
#     # 1. Loss/accuracy curves
#     plt.figure();
#     plt.plot(tr_losses, label="train_loss")
#     plt.plot(val_accs, label="val_acc")
#     plt.legend(); plt.title("Training curves"); plt.show()

#     # 2. Confusion matrix
#     cm = confusion_matrix(y_true, pred[data.test_mask.cpu().numpy()])
#     plt.figure(figsize=(6,5))
#     sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
#     plt.title("Confusion Matrix (Test)"); plt.xlabel("Pred"); plt.ylabel("True")
#     plt.show()

#     # 3. t-SNE of logits
#     logits_np = logits if isinstance(logits, np.ndarray) else logits
#     n_samples = logits_np.shape[0]
#     tsne = TSNE(n_components=2, random_state=seed, perplexity=min(5, n_samples-1))
#     emb = tsne.fit_transform(logits_np)

#     plt.figure(figsize=(6,5))
#     plt.scatter(emb[:,0], emb[:,1], c=data.y.cpu(), cmap="tab10", s=40, alpha=0.8)
#     plt.title("t-SNE of node logits")
#     plt.show()

#     return dict(test_acc=te_acc, test_loss=te_loss)

# if __name__=="__main__":
#     run_node_classification("./synthetic_graph_data", epochs=10)


In [2]:
# edu_qgc_synth_node.py
# -*- coding: utf-8 -*-
"""
Node classification on new reproducible synthetic datasets (synthetic_graph_A / synthetic_graph_B)
using EDU-QGC.
- Loads dataset from CSVs (nodes.csv, features.csv, edges.csv, metadata.json)
- Uses predefined train/val/test splits from nodes.csv
- Trains + evaluates node classification
- Visualizes + saves outputs for reproducibility
"""

import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.manifold import TSNE

import pennylane as qml
from torch_geometric.data import Data

# preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler

# =====================================================
# Reproducibility
# =====================================================
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# =====================================================
# Model (unchanged)
# =====================================================
from torch.nn import Dropout   # only add Dropout wrapper externally if needed

class EDUQGCNodeClassifier(nn.Module):
    def __init__(self, n_nodes, in_feats, T=2, seed=0, use_gpu_qnode=True, use_feat_skip=True, num_classes=7):
        super().__init__()
        self.n_nodes = n_nodes
        self.T = T
        self.use_feat_skip = use_feat_skip

        self.enc_W = nn.Parameter(torch.randn(T, 2, in_feats) * 0.08)
        self.enc_b = nn.Parameter(torch.randn(T, 2) * 0.02)

        self.edge_phase  = nn.Parameter(torch.randn(T) * 0.08)
        self.pre_theta   = nn.Parameter(torch.randn(T) * 0.08)
        self.pre_psi     = nn.Parameter(torch.randn(T) * 0.08)
        self.post_theta  = nn.Parameter(torch.randn(T) * 0.08)
        self.post_psi    = nn.Parameter(torch.randn(T) * 0.08)

        readin_dim = 1 + in_feats if use_feat_skip else 1
        self.readout = nn.Sequential(
            nn.Linear(readin_dim, num_classes),
            Dropout(p=0.3)  # 🔹 dropout for regularization
        )

        use_cuda = torch.cuda.is_available()
        qdev_name = "lightning.gpu" if (use_gpu_qnode and use_cuda) else "default.qubit"
        self.dev = qml.device(qdev_name, wires=n_nodes, shots=None)

        @qml.qnode(self.dev, interface="torch", diff_method="best")
        def circuit(edge_index, X, enc_W, enc_b,
                    edge_phase, pre_theta, pre_psi, post_theta, post_psi):
            for t in range(self.T):
                enc_out = X @ enc_W[t].T + enc_b[t]
                alphas = enc_out[:, 0]; betas = enc_out[:, 1]
                for i in range(self.n_nodes):
                    qml.RX(alphas[i], wires=i)
                    qml.RY(betas[i], wires=i)

                for i in range(self.n_nodes):
                    qml.RZ(pre_psi[t], wires=i)
                    qml.RX(pre_theta[t], wires=i)

                E = edge_index.shape[1]
                for e in range(E):
                    u = int(edge_index[0, e].item()); v = int(edge_index[1, e].item())
                    if u != v:
                        qml.ControlledPhaseShift(edge_phase[t], wires=[u, v])

                for i in range(self.n_nodes):
                    qml.RZ(post_psi[t], wires=i)
                    qml.RX(post_theta[t], wires=i)

            return [qml.expval(qml.Z(i)) for i in range(self.n_nodes)]

        self._circuit = circuit

    def forward(self, edge_index_torch, x_torch):
        device = next(self.parameters()).device
        edge_index = edge_index_torch.to(device)
        X = x_torch.to(device).float()

        out = self._circuit(edge_index, X,
                            self.enc_W, self.enc_b,
                            self.edge_phase,
                            self.pre_theta, self.pre_psi,
                            self.post_theta, self.post_psi)

        expvals = torch.stack(out, dim=0).float().to(device)
        if expvals.dim() == 1:
            expvals = expvals.unsqueeze(1)
        elif expvals.dim() == 2 and expvals.shape[1] == 1:
            pass
        else:
            expvals = expvals.squeeze(-1).unsqueeze(1)

        readin = torch.cat([expvals, X], dim=1) if self.use_feat_skip else expvals
        logits = self.readout(readin)
        return logits

# =====================================================
# Training / Evaluation
# =====================================================
@torch.no_grad()
def evaluate_nodeclf(model, data, mask):
    model.eval()
    logits = model(data.edge_index, data.x)
    pred = logits.argmax(1)
    loss = F.cross_entropy(logits[mask], data.y[mask]).item()
    acc = accuracy_score(data.y[mask].cpu(), pred[mask].cpu())
    return loss, acc, pred.cpu().numpy(), logits.detach().cpu().numpy()

# =====================================================
# Load new synthetic dataset (with features.csv)
# =====================================================
def load_synthetic_graph(dataset_dir, seed=42):
    nodes = pd.read_csv(os.path.join(dataset_dir, "nodes.csv"))
    feats = pd.read_csv(os.path.join(dataset_dir, "features.csv"))
    edges = pd.read_csv(os.path.join(dataset_dir, "edges.csv"))

    # features.csv already has f0...fN
    X = feats.drop(columns=["id"]).values.astype(np.float32)

    # normalize features
    scaler = StandardScaler()
    X = scaler.fit_transform(X).astype(np.float32)

    # labels
    labels = LabelEncoder().fit_transform(nodes["label"])
    labels = torch.tensor(labels, dtype=torch.long)

    edge_index = torch.tensor(edges.values.T, dtype=torch.long)

    data = Data(x=torch.tensor(X, dtype=torch.float),
                edge_index=edge_index,
                y=labels)

    # use provided splits
    splits = nodes["split"].tolist()
    data.train_mask = torch.tensor([s=="train" for s in splits])
    data.val_mask   = torch.tensor([s=="val" for s in splits])
    data.test_mask  = torch.tensor([s=="test" for s in splits])

    with open(os.path.join(dataset_dir, "metadata.json")) as f:
        meta = json.load(f)

    return data, meta

# =====================================================
# Run experiment
# =====================================================
from sklearn.utils.class_weight import compute_class_weight

def run_node_classification(dataset_dir, out_dir="results", epochs=30, lr=0.03, seed=42):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(out_dir, exist_ok=True)

    data, meta = load_synthetic_graph(dataset_dir)
    data = data.to(device)

    N, F, C = data.num_nodes, data.num_features, len(np.unique(data.y.cpu().numpy()))
    model = EDUQGCNodeClassifier(n_nodes=N, in_feats=F, num_classes=C).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    # class weights
    y_train = data.y[data.train_mask].cpu().numpy()
    class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    tr_losses, val_accs = [], []
    for ep in range(1, epochs+1):
        model.train()
        logits = model(data.edge_index, data.x)
        loss = loss_fn(logits[data.train_mask], data.y[data.train_mask])
        opt.zero_grad(); loss.backward(); opt.step()

        val_loss, val_acc, _, _ = evaluate_nodeclf(model, data, data.val_mask)
        tr_losses.append(loss.item()); val_accs.append(val_acc)
        print(f"Epoch {ep:03d} | tr_loss={loss.item():.3f} | val_loss={val_loss:.3f} | val_acc={val_acc:.3f}")

    # --- test ---
    te_loss, te_acc, pred, logits = evaluate_nodeclf(model, data, data.test_mask)
    print("\n[Test] loss={:.3f} acc={:.3f}".format(te_loss, te_acc))

    # Metrics
    y_true = data.y[data.test_mask].cpu().numpy()
    report = classification_report(y_true, pred[data.test_mask.cpu().numpy()], output_dict=True)
    print(classification_report(y_true, pred[data.test_mask.cpu().numpy()]))

    # --------- Save visualizations ---------
    # 1. Training curves
    plt.figure();
    plt.plot(tr_losses, label="train_loss")
    plt.plot(val_accs, label="val_acc")
    plt.legend(); plt.title("Training curves")
    plt.savefig(os.path.join(out_dir, "training_curves.png")); plt.close()

    # 2. Confusion matrix
    cm = confusion_matrix(y_true, pred[data.test_mask.cpu().numpy()])
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix (Test)"); plt.xlabel("Pred"); plt.ylabel("True")
    plt.savefig(os.path.join(out_dir, "confusion_matrix.png")); plt.close()

    # 3. t-SNE of logits
    logits_np = logits if isinstance(logits, np.ndarray) else logits
    n_samples = logits_np.shape[0]
    tsne = TSNE(n_components=2, random_state=seed, perplexity=min(5, n_samples-1))
    emb = tsne.fit_transform(logits_np)
    plt.figure(figsize=(6,5))
    plt.scatter(emb[:,0], emb[:,1], c=data.y.cpu(), cmap="tab10", s=40, alpha=0.8)
    plt.title("t-SNE of node logits")
    plt.savefig(os.path.join(out_dir, "tsne.png")); plt.close()

    # --------- Save results ---------
    results = {
        "test_loss": float(te_loss),
        "test_acc": float(te_acc),
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
        "seed": seed,
        "dataset": dataset_dir,
        "metadata": meta
    }
    with open(os.path.join(out_dir, "results.json"), "w") as f:
        json.dump(results, f, indent=2)

    return model,data,results

if __name__=="__main__":
    # # Example run on regime A
    # run_node_classification("./synthetic_graph_A", out_dir="results_A", epochs=20)
    # # Example run on regime B
    # run_node_classification("./synthetic_graph_B", out_dir="results_B", epochs=20)
    model, data, results = run_node_classification("./synthetic_graph_A", out_dir="results_A", epochs=5)
    model, data, results = run_node_classification("./synthetic_graph_B", out_dir="results_B", epochs=5)




Epoch 001 | tr_loss=1.099 | val_loss=0.861 | val_acc=0.833
Epoch 002 | tr_loss=0.918 | val_loss=0.703 | val_acc=0.833
Epoch 003 | tr_loss=0.769 | val_loss=0.588 | val_acc=0.833
Epoch 004 | tr_loss=0.419 | val_loss=0.499 | val_acc=0.833
Epoch 005 | tr_loss=0.455 | val_loss=0.432 | val_acc=0.833

[Test] loss=0.559 acc=0.857
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         2
           1       1.00      1.00      1.00         1
           2       1.00      1.00      1.00         2
           3       1.00      0.50      0.67         2

    accuracy                           0.86         7
   macro avg       0.92      0.88      0.87         7
weighted avg       0.90      0.86      0.85         7

Epoch 001 | tr_loss=1.079 | val_loss=0.904 | val_acc=0.667
Epoch 002 | tr_loss=0.936 | val_loss=0.737 | val_acc=0.833
Epoch 003 | tr_loss=0.792 | val_loss=0.620 | val_acc=0.833
Epoch 004 | tr_loss=0.678 | val_loss=0.541 | val_acc=1.000
Epoch

In [11]:
from quantum_graphlime import QuantumGraphLIME  # adjust name if needed

# Train and keep model + data
model, data, results = run_node_classification("./synthetic_graph_A", out_dir="results_A", epochs=5)

# Now run GraphLIME
model.eval()
explainer = QuantumGraphLIME(model, data, num_samples=2, num_quantum_shots=1)
node_id = 0  # choose a node
explanation = explainer.explain_node(node_id)
print(explanation)


Epoch 001 | tr_loss=1.099 | val_loss=0.861 | val_acc=0.833
Epoch 002 | tr_loss=0.918 | val_loss=0.703 | val_acc=0.833
Epoch 003 | tr_loss=0.769 | val_loss=0.588 | val_acc=0.833
Epoch 004 | tr_loss=0.419 | val_loss=0.499 | val_acc=0.833
Epoch 005 | tr_loss=0.455 | val_loss=0.432 | val_acc=0.833

[Test] loss=0.559 acc=0.857
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         2
           1       1.00      1.00      1.00         1
           2       1.00      1.00      1.00         2
           3       1.00      0.50      0.67         2

    accuracy                           0.86         7
   macro avg       0.92      0.88      0.87         7
weighted avg       0.90      0.86      0.85         7



ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)

In [ ]:
# 

Training or loading model and data...
Epoch 001 | tr_loss=1.099 | val_loss=0.861 | val_acc=0.833
Epoch 002 | tr_loss=0.918 | val_loss=0.703 | val_acc=0.833
Epoch 003 | tr_loss=0.769 | val_loss=0.588 | val_acc=0.833
Epoch 004 | tr_loss=0.419 | val_loss=0.499 | val_acc=0.833
Epoch 005 | tr_loss=0.455 | val_loss=0.432 | val_acc=0.833

[Test] loss=0.559 acc=0.857
              precision    recall  f1-score   support

           0       0.67      1.00      0.80         2
           1       1.00      1.00      1.00         1
           2       1.00      1.00      1.00         2
           3       1.00      0.50      0.67         2

    accuracy                           0.86         7
   macro avg       0.92      0.88      0.87         7
weighted avg       0.90      0.86      0.85         7

Explaining node 0...
Error during explanation: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)


: 